In [1]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

Sat Jul 11 07:54:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

📂 Verifying submodules directory contents:
total 20
drwxr-xr-x  5 root root 4096 Jul 11 07:55 .
drwxr-xr-x 10 root root 4096 Jul 11 07:55 ..
drwxr-xr-x  5 root root 4096 Jul 11 07:55 diff-gaussian-rasterization_fastgs
drwxr-xr-x  4 root root 4096 Jul 11 07:55 fused-ssim
drwxr-xr-x  3 root root 4096 Jul 11 07:55 simple-knn


Cloning into '/kaggle/working/spec-fastgs'...
Updating files: 100% (1896/1896), done.


In [3]:
%%bash
# Copy datasets from the read-only input mount into the WRITABLE working dir.
# (The synthetic loaders write points3d.ply into the scene dir, so the source MUST
#  be writable — /kaggle/input is read-only.)
WORK=/kaggle/working/spec-fastgs/spec-fastgs/datasets
mkdir -p "$WORK"

# Mip-NeRF 360 — auto-locate under /kaggle/input, trying several strategies in order,
# since the exact folder name/casing/nesting depends on how the dataset was attached.
# CONFIRMED (2026-07-05, via screenshot of the actual Kaggle Input panel): the real
# layout is spec-fastgs-datasets/datasets/datasets/mipnerf360/{bicycle,bonsai,counter,
# ...} — i.e. mipnerf360 sits 6 levels below /kaggle/input (datasets/nctuan/
# spec-fastgs-datasets/datasets/datasets/mipnerf360), one "datasets/" deeper than the
# old hardcoded path assumed. maxdepth is generously padded past that confirmed depth.
MIPNERF_SRC=""
# Strategy 1: exact name, case-insensitive, generous depth.
MIPNERF_SRC=$(find /kaggle/input -maxdepth 10 -iname "mipnerf360" -type d 2>/dev/null | head -1)
# Strategy 2: anchor on the actual scene this run needs (counter/images), in case the
# parent folder is named/nested differently than we expect.
if [ -z "$MIPNERF_SRC" ]; then
    COUNTER_IMAGES=$(find /kaggle/input -maxdepth 12 -type d -ipath "*counter/images" 2>/dev/null | head -1)
    if [ -n "$COUNTER_IMAGES" ]; then
        MIPNERF_SRC=$(dirname "$(dirname "$COUNTER_IMAGES")")
    fi
fi

if [ -n "$MIPNERF_SRC" ]; then
    echo "📥 copying $MIPNERF_SRC -> $WORK/mipnerf360"
    cp -r "$MIPNERF_SRC" "$WORK/mipnerf360"
else
    echo "⚠️  mipnerf360 NOT found under /kaggle/input — dumping the actual input"
    echo "   layout below (up to 6 levels) so the path can be fixed by hand:"
    find /kaggle/input -maxdepth 6 | sort
fi

# Synthetic suites — auto-locate under /kaggle/input regardless of the dataset slug.
for d in Anisotropic-Synthetic-Dataset Synthetic_NSVF; do
    SRC=$(find /kaggle/input -maxdepth 6 -iname "$d" -type d 2>/dev/null | head -1)
    if [ -n "$SRC" ]; then
        echo "📥 copying $SRC -> $WORK/"
        cp -r "$SRC" "$WORK/"
    else
        echo "⚠️  $d NOT found under /kaggle/input"
    fi
done
echo "📂 datasets now in working:"; ls "$WORK"

📥 copying /kaggle/input/datasets/nctuan/spec-fastgs-datasets/datasets/datasets/mipnerf360 -> /kaggle/working/spec-fastgs/spec-fastgs/datasets/mipnerf360
📥 copying /kaggle/input/datasets/nctuan/spec-fastgs-datasets/Anisotropic-Synthetic-Dataset -> /kaggle/working/spec-fastgs/spec-fastgs/datasets/
📥 copying /kaggle/input/datasets/nctuan/spec-fastgs-datasets/Synthetic_NSVF -> /kaggle/working/spec-fastgs/spec-fastgs/datasets/
📂 datasets now in working:
Anisotropic-Synthetic-Dataset
mipnerf360
Synthetic_NSVF


In [4]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

PREFIX=/opt/conda
Unpacking payload ...

Installing base environment...



Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - cudatoolkit-dev=11.7
    - gcc_linux-64=11
    - gxx_linux-64=11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    archspec-0.2.5             |     pyhd8ed1ab_0          50 KB  conda-forge
    binutils_im

In [5]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

Looking in indexes: https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 437.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 73.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.4 MB/s eta 0:00:00


In [6]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

Torch version: 1.13.1+cu117
CUDA back-end: 11.7
GPU Available: True


In [7]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

Using pip 23.3.1 from /opt/conda/lib/python3.10/site-packages/pip (python 3.10)
Processing /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for diff-gaussian-rasterization-fastgs: filename=diff_gaussian_rasterization_fastgs-0.0.0-cp310-cp310-linux_x86_64.whl size=512612 sha256=20b8c8f3b54397bf829b4471286f767535b717e949c14464981da282eb99c650
  Stored in directory: /root/.cache/pip/wheels/97/93/b8/dcb431877b1f2494a113debd5bed5c29d7742810e34742d0c6
Successfully built diff-gaussian-rasterization-fastgs
Using pip 23.3.1 from /opt/conda/lib/python3.10/site-packages/pip (python 3.10)
Processing /kaggle/working/spec-fastgs/spec-fastgs/submodules/simple-knn
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for simple-knn: filename=simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl

  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info
  writing /tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/SOURCES.txt'
  adding license file 'LICENSE.md'
  writing manifest file '/tmp/pip-pip-egg-info-21sgses_/diff_gaussian_rasterization_fastgs.egg-info/SOURCES.txt'
  Running command python setup.py bdist_wheel
  running bdist_wheel
  running build
  running build_py
  creating build
  creating build/lib

In [8]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

✅ FastGS CUDA extensions compiled and loaded successfully!


In [9]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

running bdist_wheel
running build
running build_py
running build_ext
building 'diff_gaussian_rasterization_fastgs._C' extension
ninja: no work to do.
g++-11 -shared -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/build/temp.linux-x86_64-cpython-310/cuda_rasterizer/adam.o /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/build/temp.linux-x86_64-cpython-310/cuda_rasterizer/backward.o /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/build/temp.linux-x86_64-cpython-310/cuda_rasterizer/forward.o /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/build/temp.linux-x86_64-cpython-310/cuda_rasterizer/rasterizer_impl.o /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/bui

Emitting ninja build file /kaggle/working/spec-fastgs/spec-fastgs/submodules/diff-gaussian-rasterization_fastgs/build/temp.linux-x86_64-cpython-310/build.ninja...
Compiling objects...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
/opt/conda/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
Emitting ninja build file /kaggle/working/spec-fastgs/spec-fastgs/submodules/simple-knn/build/temp.linux-x86_64-cpython-3

In [10]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

✨ Wheels safely compiled and extracted to: /kaggle/working/fastgs_wheels_py310
total 1256
drwxr-xr-x 2 root root   4096 Jul 11 08:08 .
drwxr-xr-x 4 root root   4096 Jul 11 08:08 ..
-rw-r--r-- 1 root root 512612 Jul 11 08:08 diff_gaussian_rasterization_fastgs-0.0.0-cp310-cp310-linux_x86_64.whl
-rw-r--r-- 1 root root 203102 Jul 11 08:08 fused_ssim-0.0.0-cp310-cp310-linux_x86_64.whl
-rw-r--r-- 1 root root 553927 Jul 11 08:08 simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl


In [11]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm imageio

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of plyfile to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.2/186.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 27.1 MB/s eta 0:00:00


In [12]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

🎉 All systems functional and ready for execution!


In [13]:
# ============================================================
# ⚙️ EXPERIMENT CONFIGURATION
# ============================================================
# Choose scenes/datasets and image resolution scales to run on.
# Mip-NeRF 360 scenes: "counter", "bicycle", "bonsai", "kitchen", "room", "garden", "flowers", "treehill", "stump"
# Image resolution scales: "images" (original), "images_2" (1/2), "images_4" (1/4), "images_8" (1/8)
# Shiny/Synthetic scenes: "toaster"

SCENES = ["counter"]
IMAGES_LIST = ["images_8"]

import os
os.environ['SCENES'] = " ".join(SCENES)
os.environ['IMAGES_LIST'] = " ".join(IMAGES_LIST)

print(f"✅ Experiment Configured:")
print(f"   SCENES      = {SCENES}")
print(f"   IMAGES_LIST = {IMAGES_LIST}")


✅ Experiment Configured:
   SCENES      = ['counter']
   IMAGES_LIST = ['images_8']


In [14]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP PREREQUISITE -- extract the Reflection Prior (Shafer/Klinker,
# extract_reflection_prior.py) ONCE for each selected scene/images combination.
# ============================================================
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd /kaggle/working/spec-fastgs/spec-fastgs

for SCENE_NAME in $SCENES; do
    for IMAGE_SCALE in $IMAGES_LIST; do
        echo "============================================================"
        echo "📥 Extracting Reflection Prior: SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
        echo "============================================================"
        
        # Dataset layout guard (idempotent)
        if [ -d "./datasets/datasets" ]; then
            echo "Re-aligning dataset file structure..."
            mv ./datasets/datasets/* ./datasets/
            rm -rf ./datasets/datasets
        fi

        if [ ! -d "./datasets/mipnerf360/${SCENE_NAME}/${IMAGE_SCALE}" ] && [ ! -d "./datasets/mipnerf360/${SCENE_NAME}/images" ]; then
            echo "❌ ./datasets/mipnerf360/${SCENE_NAME}/${IMAGE_SCALE}(or images) not found."
            continue
        fi
        
        python extract_reflection_prior.py \
            -s ./datasets/mipnerf360/${SCENE_NAME} \
            -i ${IMAGE_SCALE} \
            --sk_intensity 0.7 \
            --sk_saturation 0.2

        echo "reflection priors written:"
        ls ./datasets/mipnerf360/${SCENE_NAME}/reflection_prior | head -5
        echo "... total:" $(ls ./datasets/mipnerf360/${SCENE_NAME}/reflection_prior/*.png | wc -l) "png priors"
    done
done


📥 Extracting Reflection Prior: SCENE=counter, IMAGES=images_8
Reading camera 240/240 [11/07 08:08:23]
Loading Training Cameras [11/07 08:08:23]
Loading Test Cameras [11/07 08:08:27]
Number of points at initialisation :  155767 [11/07 08:08:27]
Loaded 240 training cameras. [11/07 08:08:27]
Output directory: /kaggle/working/spec-fastgs/spec-fastgs/datasets/mipnerf360/counter/reflection_prior [11/07 08:08:27]
Prior extraction complete in 9.07 seconds! [11/07 08:08:31]
reflection priors written:
DSCF5857_ref_score.png
DSCF5858_ref_score.png
DSCF5859_ref_score.png
DSCF5860_ref_score.png
DSCF5861_ref_score.png
... total: 240 png priors


Extracting Priors (tan): 100%|██████████| 240/240 [00:03<00:00, 72.72it/s]


In [ ]:
%%bash
# ============================================================
# ASG_DEGREE SWEEP, PART 1/2 -- USE_REF_SCORE=True, Mip-NeRF 360
# Runs ASG_DEGREE=32 on GPU 0 and ASG_DEGREE=48 on GPU 1 AT THE SAME TIME.
# ============================================================
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

for SCENE_NAME in $SCENES; do
    for IMAGE_SCALE in $IMAGES_LIST; do
        echo "============================================================"
        echo "🚀 Running Sweep (ASG 32/48): SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
        echo "============================================================"

        if [ -d "./datasets/datasets" ]; then
            mv ./datasets/datasets/* ./datasets/
            rm -rf ./datasets/datasets
        fi

        # EXTRACT_REF_PRIOR=False on both legs: the reflection prior was already
        # generated once for this scene/scale in the cell above and does not depend
        # on ASG_DEGREE, so regenerating it here would just race two processes
        # mv-ing/rewriting the same ./datasets/mipnerf360/<scene>/reflection_prior
        # directory at the same time.
        echo "=== launching ASG_DEGREE=32 on GPU 0 ==="
        CUDA_VISIBLE_DEVICES=0 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=32 USE_REF_SCORE=True \
            EXTRACT_REF_PRIOR=False \
            OUTPUT_SUFFIX=_asg32_ref \
            bash run_spec-fastgs_big.sh > /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log 2>&1 &
PID32=$!

        echo "=== launching ASG_DEGREE=48 on GPU 1 ==="
        CUDA_VISIBLE_DEVICES=1 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=48 USE_REF_SCORE=True \
            EXTRACT_REF_PRIOR=False \
            OUTPUT_SUFFIX=_asg48_ref \
            bash run_spec-fastgs_big.sh > /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log 2>&1 &
PID48=$!

        echo "both launched (pid32=$PID32, pid48=$PID48) -- waiting for both to finish..."
        wait $PID32
        STATUS32=$?
        wait $PID48
        STATUS48=$?

        echo "--- tail of ${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log ---"; tail -n 40 /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg32_ref.log
        echo "--- tail of ${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log ---"; tail -n 40 /kaggle/working/${SCENE_NAME}_${IMAGE_SCALE}_asg48_ref.log

        echo "ASG_DEGREE=32 exit status: $STATUS32"
        echo "ASG_DEGREE=48 exit status: $STATUS48"
        if [ $STATUS32 -ne 0 ] || [ $STATUS48 -ne 0 ]; then
            echo "⚠️  sweep failed for ${SCENE_NAME} ${IMAGE_SCALE} -- check logs at /kaggle/working/"
            exit 1
        fi
    done
done


In [ ]:
# %%bash
# # ============================================================
# # ASG_DEGREE SWEEP, PART 2/2 -- USE_REF_SCORE=True, Mip-NeRF 360
# # ASG_DEGREE=64, run sequentially for all combinations.
# # ============================================================
# set -e
# export PATH=/opt/conda/bin:$PATH
# source /opt/conda/bin/activate
# export CUDA_HOME=/opt/conda
# export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH


# for SCENE_NAME in $SCENES; do
#     for IMAGE_SCALE in $IMAGES_LIST; do
#         echo "============================================================"
#         echo "🚀 Running ASG_DEGREE=64: SCENE=${SCENE_NAME}, IMAGES=${IMAGE_SCALE}"
#         echo "============================================================"
#         CUDA_VISIBLE_DEVICES=0 SCENE=${SCENE_NAME} IMAGES=${IMAGE_SCALE} ASG_DEGREE=64 USE_REF_SCORE=True \
#             EXTRACT_REF_PRIOR=False \
#             OUTPUT_SUFFIX=_asg64_ref \
#             bash run_spec-fastgs_big.sh
#     done
# done


In [ ]:
import shutil, os

# Automatically determine the active scenes from the environment
scenes_env = os.environ.get('SCENES', 'counter')
scenes = scenes_env.split()

runs = []
for active_scene in scenes:
    runs.extend([
        (f"{active_scene}_asg32_ref", f"spec_fastgs_output_{active_scene}_asg32_ref"),
        (f"{active_scene}_asg48_ref", f"spec_fastgs_output_{active_scene}_asg48_ref"),
        (f"{active_scene}_asg64_ref", f"spec_fastgs_output_{active_scene}_asg64_ref"),
    ])

for scene_dir, out_name in runs:
    src = f"/kaggle/working/spec-fastgs/spec-fastgs/output/{scene_dir}"
    out = f"/kaggle/working/{out_name}"
    if os.path.isdir(src):
        shutil.make_archive(out, "zip", src)
        print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
    else:
        print(f"no {scene_dir} output found at", src)
